In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt


In [3]:
df = pd.read_csv("modelling (1).csv")

## Feature Engineering

In the Rossman Sales dataset PromoInterval tells us which months recurring promotion (promo 2) is running. 
If a store participates in Promo2 (meaning the Promo2 column is 1), it doesn't run the promotion every single day of the year. Instead, the promotion runs in a 3-month repeating cycle. 

IsPromo2Active- tells your model whether a store's recurring promotion (Promo2) is active on a particular date. 
-equals 1 when the store participates in Promo2, the programme has started, and the month is listed in PromoInterval. Otherwise, it equals 0. 


In [4]:


df['Date'] = pd.to_datetime(df['Date'])

# Months in which the store starts a new Promo2 promotion
df['IsPromo2Active'] = df.apply(
    lambda row: int(
        row['Promo2'] == 1
        and pd.notna(row['PromoInterval'])
        and row['Date'].strftime('%b') in row['PromoInterval'].split(',')
        and row['Date'].isocalendar().year * 100
            + row['Date'].isocalendar().week
            >= row['Promo2SinceYear'] * 100
            + row['Promo2SinceWeek']
    ) if pd.notna(row['Promo2SinceYear'])
        and pd.notna(row['Promo2SinceWeek'])
        else 0,
    axis=1
)


In [6]:
df['IsPromo2Active'].unique()

array([0, 1])

Day 7 is adjacent to day 1 of the week (eg. monday is adjacent to sunday but the numbers make it seem like they are far apart)
So using sin/cos of month/dayofweek makes the month/dayofweek cyclic 
sin and cos are used because for some values sin is 0 and for those values sin is not 0
This may imporve the performance of an mlp 

In [5]:
# Day of week: 7-day cycle
df['DayOfWeek_sin'] = np.sin(2 * np.pi * df['DayOfWeek'] / 7)
df['DayOfWeek_cos'] = np.cos(2 * np.pi * df['DayOfWeek'] / 7)

# Month: 12-month cycle
df['Month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)

In [7]:
X = df[['DayOfWeek_sin',
    'DayOfWeek_cos',
    'Month_sin',
    'Month_cos',
    'Open',
    'Promo',
    'StateHoliday',
    'SchoolHoliday',
    'StoreType',
    'Assortment',
    'CompetitionDistance',
    'CompetitionOpenSinceMonth',
    'CompetitionOpenSinceYear',
    'Promo2',
    'Promo2SinceWeek',
    'Promo2SinceYear',
    'Year',
    'Month',
    'Day',
    'WeekOfYear',
       'IsPromo2Active']]
Y = df['Sales']

from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.25, random_state = 0)



In [8]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
continuous_cols = ['DayOfWeek_sin',
    'DayOfWeek_cos',
    'Month_sin',
    'Month_cos',
    'Open',
    'Promo',
    'StateHoliday',
    'SchoolHoliday',
    'StoreType',
    'Assortment',
    'CompetitionDistance',
    'CompetitionOpenSinceMonth',
    'CompetitionOpenSinceYear',
    'Promo2',
    'Promo2SinceWeek',
    'Promo2SinceYear',
    'Year',
    'Month',
    'Day',
    'WeekOfYear',
    'IsPromo2Active']
X_train[continuous_cols] = sc.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = sc.transform(X_test[continuous_cols])

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
mlp = MLPRegressor(
 hidden_layer_sizes= (20,20),
 activation = 'relu',
 solver = 'adam',
 max_iter = 5000,
 random_state = 0
)
mlp.fit(X_train, Y_train)

In [ ]:
Y_pred_test = mlp.predict(X_test)
Y_pred_train = mlp.predict(X_train)

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

In [ ]:
# 2. Evaluate Training Set Metrics
train_mae = mean_absolute_error(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_r2 = r2_score(y_train, y_train_pred)


# 3. Evaluate Testing Set Metrics
test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_r2 = r2_score(y_test, y_test_pred)

In [ ]:
print("--- Training Metrics ---")
print(f"MAE : {train_mae}")
print(f"RMSE: {train_rmse}")
print(f"R2  : {train_r2}\n")

print("--- Testing Metrics ---")
print(f"MAE : {test_mae}")
print(f"RMSE: {test_rmse}")
print(f"R2  : {test_r2}")